In [1]:
from datasets import load_dataset, load_from_disk
import polars as pl
import pandas as pd
import json
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader
import faiss
from functools import reduce
import datasets
import torch
import torch.nn as nn
from tqdm import tqdm
import os
from datetime import datetime

NUM_PROC = 32
CACHE_DIR = "/home/jupyter/filestore/storage/"

/usr/local/lib/python3.10/dist-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/usr/local/lib/python3.10/dist-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
2026-05-01 21:30:03.326155: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-01 21:30:06.917501: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
DATA_PATH = "/home/jupyter/filestore/storage/datasets/user_events_20230501"

dataset = load_from_disk(DATA_PATH)

In [3]:
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(-1)

polars.config.Config

In [4]:
polars_ds = dataset.to_polars()

In [5]:
with open("data/color2id", "rb") as fp:
    color2id = pickle.load(fp)
    
with open("data/condition2id", "rb") as fp:
    condition2id = pickle.load(fp)
    
with open("data/size2id", "rb") as fp:
    size2id = pickle.load(fp)

In [6]:
TRAIN_END_DT = pd.to_datetime("2023-05-14")
TEST_END_DT = pd.to_datetime("2023-05-21")

items = (
    polars_ds
    .with_columns(pl.col("stime").cast(pl.Date).alias("date"))
    .filter(pl.col("date") <= TEST_END_DT)
    .with_columns(
        pl.col("item_condition_name").fill_null("NO_INFO"),
        pl.col("size_name").fill_null("NO_INFO"),
        pl.col("color").fill_null("NO_INFO")
    )
    .select("item_id", "name", "price", "item_condition_name", "size_name", "color")
    .unique()
    .with_columns(
        pl.col("item_condition_name").apply(lambda x: condition2id[x]),
        pl.col("size_name").apply(lambda x: size2id[x]),
        pl.col("color").apply(lambda x: color2id[x])
    )
    .groupby("item_id")
    .agg(pl.struct("name", "price", "item_condition_name", "size_name", "color").alias("item_attrs"))
)

In [7]:
items.head(10)

item_id,item_attrs
i64,list[struct[5]]
221405120,"[{""Polk System Subwoofer"",200.0,1,146,708}]"
20888256,"[{""Official Don’t Starve Chester Plush NEW"",10.0,1,146,708}]"
207293760,"[{""Fiestaware Dinner dish and salad dish"",30.4,0,146,708}]"
139088288,"[{""Y2K Rock and Republic jeans size 24"",30.0,0,80,708}]"
167724896,"[{""monster high chucky and tiffany"",125.0,1,146,708}]"
69809408,"[{""Cocktail rings lot"",45.0,0,146,708}]"
160631392,"[{""Sorel and Fern Baby Mobile Safari"",25.0,2,146,708}]"
106944960,"[{""Wedding dress"",98.0,0,112,708}]"
60073568,"[{""PORTUGESE CORK BAG"",67.69,1,146,708}]"


In [12]:
class InferenceDataset(Dataset):
    def __init__(self, items):
        self.items = items
        
    def __len__(self):
        return self.items.shape[0]
    
    def __getitem__(self, idx):
        item_attrs = self.items["item_attrs"][idx][0]
        return item_attrs

In [26]:
inference_dataset = InferenceDataset(items)
inference_loader = DataLoader(inference_dataset, batch_size=2048)

In [14]:
inference_dataset[0]

{'name': 'Polk System Subwoofer',
 'price': 200.0,
 'item_condition_name': 1,
 'size_name': 146,
 'color': 708}

In [27]:
class ItemEmbedder(nn.Module):
    def __init__(self, text_encoder, price_quantiles, condition_num_values, size_num_values, color_num_values, emb_dim=64, device="cpu"):
        super().__init__()
        self.text_encoder = text_encoder
        self.device = device
        self.bins = torch.tensor(price_quantiles).to(device)
        self.bin_emb = nn.Embedding(len(price_quantiles), emb_dim)
        self.condition_emb = nn.Embedding(condition_num_values, 32)
        self.size_emb = nn.Embedding(size_num_values, 64)
        self.color_emb = nn.Embedding(color_num_values, 128)
        self.lin1 = nn.Linear(384 + 64 + 32 + 64 + 128, 256)
        self.lin2 = nn.Linear(256, 128)
        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        
        
    def price_encoder(self, prices):
        B = prices.shape[0]
        idx = torch.bucketize(prices, self.bins, right=True)
        idx = torch.where(idx < len(self.bins), idx, idx - 1)
        left_border = self.bins[idx - 1]
        right_border = self.bins[idx]
        bins_len = right_border - left_border
        left_weights = (1 - (prices - left_border) / bins_len).reshape(B, 1)
        right_weights = (1 - (right_border - prices) / bins_len).reshape(B, 1)
        left_emb = left_weights * self.bin_emb(idx - 1)
        right_emb = right_weights * self.bin_emb(idx)
        price_emb = left_emb + right_emb
        return price_emb
    
    def forward(self, x):
        name_emb = self.text_encoder.encode(x["name"], batch_size=1024, normalize_embeddings=True, convert_to_tensor=True)
        price_emb = self.price_encoder(x["price"])
        condition_emb = self.condition_emb(x["item_condition_name"])
        size_emb = self.size_emb(x["size_name"])
        color_emb = self.color_emb(x["color"])
        item_emb = torch.cat([name_emb, price_emb, condition_emb, size_emb, color_emb], dim=-1).to(torch.float32)
        out_emb = self.lin2(self.relu(self.lin1(item_emb)))
        return out_emb

In [28]:
price_quantiles = [1.0, 9.0, 12.0, 15.0, 20.0, 25.0, 34.0, 46.0, 70.0, 133.0, 5000.0]

In [29]:
text_encoder = SentenceTransformer("all-MiniLM-L6-v2")

for p in text_encoder.parameters():
    p.requires_grad = False
    
model = ItemEmbedder(text_encoder, price_quantiles, len(condition2id.keys()), len(size2id.keys()), len(color2id.keys()), device="cuda:0")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 710.57it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [30]:
model.load_state_dict(torch.load("data/item2item_v2.pth", weights_only=True))

<All keys matched successfully>

In [31]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"

all_embs = []
item_ids = items["item_id"].to_list()

model = model.to(device)
with torch.no_grad():
    for batch in tqdm(inference_loader, total=len(inference_loader), desc="inference"):
        for k in batch.keys():
            if k != "name":
                batch[k] = batch[k].to(device)
        embs = model(batch)
        normalized_embs = torch.nn.functional.normalize(embs, p=2.0, dim=-1).detach().cpu().numpy()
        all_embs.append(normalized_embs)
    
all_embs = np.concatenate(all_embs, axis=0)

inference: 100%|██████████| 2988/2988 [11:11<00:00,  4.45it/s]


TypeError: concatenate() got an unexpected keyword argument 'dim'

In [42]:
np.save("data/item_embeddings", all_embs)

In [33]:
all_embs = np.concatenate(all_embs, axis=0)

In [34]:
all_embs.shape

(6117994, 128)

In [41]:
with open("data/item_ids", "wb") as fp:
    pickle.dump(item_ids, fp)

In [39]:
m = 16
nbits = 8
dim = 128
n_clusters = 2500

quantizer = faiss.IndexFlatIP(dim)

ivfpq = faiss.IndexIVFPQ(
    quantizer,
    dim,
    n_clusters,
    m,
    nbits,
    faiss.METRIC_INNER_PRODUCT
)

opq = faiss.OPQMatrix(dim, m)
index = faiss.IndexPreTransform(opq, ivfpq)

index.train(all_embs)

index.add(all_embs)

ivf = faiss.extract_index_ivf(index)
ivf.nprobe = 64
ivfpq.use_precomputed_table = True

In [40]:
faiss.write_index(index, "data/neural_index.faiss")